# Synthetic pipeline: certified SIIT models → PLT/CLT → attribution graphs → depth
Runtime → Change runtime type → **T4 GPU**. Results are written to Google Drive, and every stage resumes after a disconnect.

The reported results were produced in three parts with identical settings: **A** (D = 1–3), **B** (D = 4), **C** (D = 5). Choose one below.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import glob, os, sys
zips = glob.glob('/content/clt-depth-collapse*.zip')
if zips:                                   # option A: uploaded zip of this repository
    !unzip -q -o "{zips[0]}" -d /content/
else:                                      # option B: clone from GitHub
    !git clone -q https://github.com/sirikrishna-dev/clt-depth-collapse.git /content/clt-depth-collapse
os.chdir('/content/clt-depth-collapse')
!pip -q install -r requirements.txt
!python -m pytest -q tests

In [ ]:
PART = 'A'   # 'A' (D=1-3), 'B' (D=4) or 'C' (D=5)
CFG = {'A': 'configs/part_A_D1-3.yaml', 'B': 'configs/part_B_D4.yaml', 'C': 'configs/part_C_D5.yaml'}[PART]
D = f'/content/drive/MyDrive/clt_results/part_{PART}'
os.makedirs(D + '/results', exist_ok=True); os.makedirs(D + '/checkpoints', exist_ok=True)
!rm -rf results/part_{PART} checkpoints/part_{PART}; mkdir -p results checkpoints
!ln -s {D}/results results/part_{PART}; ln -s {D}/checkpoints checkpoints/part_{PART}

**1. Train and certify the SIIT models.** Running this cell again retrains failed models with 2× and then 4× steps (at most 2 retries).

In [ ]:
!python scripts/run_pipeline.py --config {CFG} --stage zoo

**2. Matched-budget replacements (PLT, CLT, parameter-matched PLT), 3. attribution graphs, 4. validation.**

In [ ]:
!python scripts/run_pipeline.py --config {CFG} --stage replacements
!python scripts/run_pipeline.py --config {CFG} --stage graphs
!python scripts/run_pipeline.py --config {CFG} --stage validate
!python scripts/run_pipeline.py --config {CFG} --stage probes
!python scripts/logit_bypass.py --results results/part_{PART} --ckpt checkpoints/part_{PART}

**5. After all three parts:** merge and build the tables and figures.

In [ ]:
!python scripts/merge_results.py results/part_A results/part_B results/part_C --out results/merged
!python scripts/make_tables.py --results results/merged --paper paper/generated
!python scripts/make_figures.py --results results/merged --out paper/figures